In [0]:
from pyspark.sql.functions import col, when, isnan

# =========================
# LOAD
# =========================
df = spark.read.table("iotmlhealthcatalog.silver.vitaldbtrain")

# =========================
# DERIVED FEATURES
# =========================
df = df \
    .withColumn(
        "shock_index",
        when(col("sbp") > 0, col("hr") / col("sbp")).otherwise(0)
    ) \
    .withColumn(
        "hr_spo2_ratio",
        when(col("spo2") > 0, col("hr") / col("spo2")).otherwise(0)
    ) \
    .withColumn(
        "is_low_sbp",
        when(col("sbp") < 90, 1.0).otherwise(0.0)
    ) \
    .withColumn(
        "is_high_hr",
        when(col("hr") > 100, 1.0).otherwise(0.0)
    ) \
    .withColumn(
        "is_low_spo2",
        when(col("spo2") < 92, 1.0).otherwise(0.0)
    )

# =========================
# FEATURE LIST
# =========================
features_list = [
    "sbp",
    "hr",
    "spo2",
    "temp",
    "shock_index",
    "hr_spo2_ratio",
    "is_low_sbp",
    "is_high_hr",
    "is_low_spo2"
]

# =========================
# CLEAN (sécurisation)
# =========================
for c in features_list:
    df = df.withColumn(
        c,
        when(isnan(col(c)) | col(c).isNull(), 0).otherwise(col(c))
    )

# =========================
# FINAL SELECT
# =========================
gold_df = df.select(
    "caseid",
    "timestamp",
    "target",
    *features_list
)

# =========================
# SAVE
# =========================
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("iotmlhealthcatalog.gold.vitaldbtrain")

print("✅ GOLD TRAIN ALIGNED WITH STREAMING")
print("✅ GOLD TRAINING TABLE CREATED")